# R1-Distill-Llama-8B runs on Kaggle (sub-team B)

Runs the experiment grid for `deepseek-ai/DeepSeek-R1-Distill-Llama-8B` with vLLM
(proposal 4.3). Before running:

1. The Kaggle account has **phone verification** done (Settings), or GPUs and internet stay off.
2. Right-hand panel: **Accelerator: GPU T4 x2**, **Internet: on**.
3. Don't start the main run until `make_conditions.py --require-reviewed` passes on `main`
   (the Malayalam review is the last one pending).

A session can run for 12 hours and the account gets about 30 GPU hours a week, so the main
run stops itself after `MAX_HOURS` and continues from where it stopped the next time you run
this notebook (with the logs from earlier sessions put back in `runs/`, see the last cell).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Set-up

In [ ]:
BRANCH = "Team-B"     # switch to "main" once Team-B is merged
TAG = "kaggle"        # log file suffix: runs/r1-distill-llama-8b/<date>-kaggle.jsonl
MAX_HOURS = 11        # stop before Kaggle's 12-hour session limit

!git clone -q --branch $BRANCH https://github.com/modestassumption/Audit-on-AI-thought-processing.git
%cd Audit-on-AI-thought-processing
!git log --oneline -1

In [ ]:
# Installs vLLM (and the torch build it needs). If this breaks Kaggle's PyTorch, pin an older vLLM
# here, e.g. !pip install -q "vllm==0.10.2", and record the version that works in pyproject.toml.
!pip install -q -e ".[kaggle]"
!python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| GPUs', torch.cuda.device_count())"

## Earlier sessions' logs

Runs already logged are skipped, so put the earlier logs back first. Either they are already on the
branch you cloned (pushed from your machine), or add your downloaded `runs_r1.zip` as a Kaggle
dataset and unzip it here.

In [ ]:
# !unzip -o -q /kaggle/input/<your-dataset>/runs_r1.zip -d .
!ls -la runs/r1-distill-llama-8b 2>/dev/null || echo "no earlier logs"

## Pilot: 50 questions

Same 50 questions as the Groq pilot. About 1,150 runs; check the extraction table before the main run.

In [ ]:
!python scripts/run_vllm.py --model r1-distill-llama-8b --tp 2 --dtype half --n-questions 50 --tag $TAG --max-hours $MAX_HOURS

In [ ]:
!python scripts/check_extraction.py --models r1-distill-llama-8b --sample 10

## Main run: all 300 questions

Picks up after the pilot (those runs are already logged). If loading the model runs out of GPU
memory, add `--gpu-memory-utilization 0.85` or `--enforce-eager`.

In [ ]:
!python scripts/run_vllm.py --model r1-distill-llama-8b --tp 2 --dtype half --tag $TAG --max-hours $MAX_HOURS

In [ ]:
!python scripts/check_extraction.py --models r1-distill-llama-8b
!python scripts/report_runs.py --models r1-distill-llama-8b --out results/r1_report.md

## Save the logs before the session ends

Download `runs_r1.zip` from the Output panel, unzip it into your clone, then commit and push
`runs/r1-distill-llama-8b/`. Add the GPU hours printed by `run_vllm.py` to `docs/compute.md`.

In [ ]:
!zip -q -r /kaggle/working/runs_r1.zip runs/r1-distill-llama-8b results
!ls -la /kaggle/working/runs_r1.zip